# Efficient and Reproducible Biomedical Question Answering using RAG
## Reproduction of IEEE SDS 2025 Paper

---

# 1. Install Dependencies

---

# 2. Import Libraries

---

# 3. Load BioASQ Dataset

---

# 4. Explore the Dataset

In [14]:
!pip install datasets
!pip install pandas
!pip install numpy
!pip install matplotlib

In [15]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
!git clone https://github.com/slinusc/medical_RAG_system.git

Cloning into 'medical_RAG_system'...
remote: Enumerating objects: 1067, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 1067 (delta 37), reused 9 (delta 4), pack-reused 997 (from 1)
Receiving objects: 100% (1067/1067), 39.07 MiB | 11.00 MiB/s, done.
Resolving deltas: 100% (566/566), done.


In [18]:
%cd medical_RAG_system

/content/medical_RAG_system


In [19]:
!pwd

/content/medical_RAG_system


In [20]:
!ls

evaluation	       LICENSE	   README.md	     sys_requirements.txt
information_retrieval  rag_system  requirements.txt


In [21]:
!head -100 README.md


# Medical RAG System

This repository contains a comprehensive implementation of a Medical Retrieval-Augmented Generation (RAG) system. The system integrates multiple components for document retrieval, question answering, and evaluation, tailored specifically for the medical domain.

## Table of Contents
- [Overview](#overview)
- [File Structure](#file-structure)
- [Installation](#installation)
- [Usage](#usage)
- [Components](#components)
  - [Retrieval System](#retrieval-system)
  - [Question Answering System](#question-answering-system)
  - [Evaluation](#evaluation)
  - [Data Storage](#data-storage)
- [Contributing](#contributing)
- [License](#license)

## Overview

The Medical RAG System is designed to enhance medical information retrieval and provide accurate answers to medical queries. It combines various retrieval methods, including BM25, bioBERT, and hybrid models, with advanced question-answering techniques to ensure precise and relevant results.


## File structure

```plain

In [22]:
!cat requirements.txt

anaconda==0.0.1.1  # Anaconda package
annotated-types==0.6.0  # Support for typing-annotations
anyio==4.3.0  # Async network and file operations
argon2-cffi==23.1.0  # The secure Argon2 password hashing algorithm
attrs==23.2.0  # Attributes without boilerplate
Babel==2.14.0  # Internationalization utilities
beautifulsoup4==4.12.3  # Screen-scraping library
bleach==6.1.0  # Sanitize your inputs
click==8.1.7  # Command Line Interface Creation Kit
decorator==5.1.1  # Simplifies the usage of decorators
elastic-transport==8.13.0  # Transport layer for Elasticsearch
elasticsearch==8.13.0  # Official Elasticsearch client
faiss-cpu==1.8.0  # A library for efficient similarity search and clustering
Flask==3.0.3  # Micro web framework
fsspec==2024.3.1  # File system specification
huggingface-hub==0.22.2  # Client library for Huggingface hub
idna==3.3  # Internationalized Domain Names in Applications (IDNA)
importlib-metadata==4.6.4  # Library to access the metadata for a Python package
joblib==1

In [23]:
import torch

print("PyTorch Version :", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("No GPU Found!")

PyTorch Version : 2.11.0+cu128
CUDA Available : True
GPU : Tesla T4


In [24]:
!pip install -q \
sentence-transformers==2.7.0 \
transformers==4.40.0 \
faiss-cpu==1.8.0 \
rank_bm25 \
datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 61.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [25]:
!pip install -q datasets

In [26]:
from datasets import load_dataset

In [27]:
pubmed = load_dataset(
    "slinusc/PubMedAbstractsSubset",
    split="train",
    streaming=True
)

README.md:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

In [28]:
first_doc = next(iter(pubmed))

In [29]:
print(first_doc.keys())

dict_keys(['title', 'abstract', 'PMID'])


In [30]:
print("PMID:")
print(first_doc["PMID"])

print("\nTitle:")
print(first_doc["title"])

print("\nAbstract:")
print(first_doc["abstract"])

PMID:
22

Title:
[Demonstration of tumor inhibiting properties of a strongly immunostimulating low-molecular weight substance. Comparative studies with ifosfamide on the immuno-labile DS carcinosarcoma. Stimulation of the autoimmune activity for approx. 20 days by BA 1, a N-(2-cyanoethylene)-urea. Novel prophylactic possibilities].

Abstract:
A report is given on the recent discovery of outstanding immunological properties in BA 1 [N-(2-cyanoethylene)-urea] having a (low) molecular mass M = 111.104. Experiments in 214 DS carcinosarcoma bearing Wistar rats have shown that BA 1, at a dosage of only about 12 percent LD50 (150 mg kg) and negligible lethality (1.7 percent), results in a recovery rate of 40 percent without hyperglycemia and, in one test, of 80 percent with hyperglycemia. Under otherwise unchanged conditions the reference substance ifosfamide (IF) -- a further development of cyclophosphamide -- applied without hyperglycemia in its most efficient dosage of 47 percent LD50 (150

In [31]:
!find . -type f | grep -i "bioasq"

In [32]:
!find . -type f | grep -i "json"

./information_retrieval/elastic_container/errors.jsonl


In [33]:
!grep -Rin "BioASQ" .

./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:185:    "rag_type.analyze_performance(\"/home/ubuntu/questions_answers_data/experiment_results/experiment_bioASQ_min2/result_ragver_4.json\")"
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:204:      "Directory 'experiment_bioASQ_min1_bioBERT' already exists at /home/ubuntu/questions_answers_data/experiment_results/experiment_bioASQ_min1_bioBERT\n"
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:209:    "experiment_name = \"experiment_bioASQ_min1_bioBERT\"\n",
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:213:    "question_input_factoid = \"/home/ubuntu/questions_answers_data/bioASQ_data_min_1/factoid_questions.json\"\n",
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:214:    "question_input_summary = \"/home/ubuntu/questions_answers_data/bioASQ_data_min_1/summary_questions.json\"\n",
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:215:    "question_input_list = \

In [34]:
!grep -Rin "bioasq" .

./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:185:    "rag_type.analyze_performance(\"/home/ubuntu/questions_answers_data/experiment_results/experiment_bioASQ_min2/result_ragver_4.json\")"
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:204:      "Directory 'experiment_bioASQ_min1_bioBERT' already exists at /home/ubuntu/questions_answers_data/experiment_results/experiment_bioASQ_min1_bioBERT\n"
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:209:    "experiment_name = \"experiment_bioASQ_min1_bioBERT\"\n",
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:213:    "question_input_factoid = \"/home/ubuntu/questions_answers_data/bioASQ_data_min_1/factoid_questions.json\"\n",
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:214:    "question_input_summary = \"/home/ubuntu/questions_answers_data/bioASQ_data_min_1/summary_questions.json\"\n",
./evaluation/evaluation_QA_system/evaluation_pipeline.ipynb:215:    "question_input_list = \

In [35]:
!grep -Rin "questions" .

./rag_system/openAI_chat.py:17:            "documents to answer questions. The first documents should be the most relevant."
./rag_system/openAI_chat.py:19:            "When answering questions, always format your response "
./rag_system/openAI_chat.py:22:            "Please think step-by-step before answering questions and provide the most accurate response possible."
./rag_system/pipeline.ipynb:497:      "        \"content\": \"Attention Deficit Hyperactivity Disorder (ADHD) is a common neurobehavioral problem in children that the medical practitioner is frequently asked to diagnose and treat. Equally as important as accurate diagnosis and treatment, however, is the ability to provide family members with clear and concise information that leads to an understanding of the disorder. This article presents a framework for answering family members' specific questions about ADHD and recommendations for ways to effectively share information with families regarding ADHD.\",\n",
./README.md:4

In [36]:
!jupyter nbconvert \
--to script \
evaluation/evaluation_QA_system/dataset_filter/filter_data.ipynb

[NbConvertApp] Converting notebook evaluation/evaluation_QA_system/dataset_filter/filter_data.ipynb to script
[NbConvertApp] Writing 20655 bytes to evaluation/evaluation_QA_system/dataset_filter/filter_data.py


In [37]:
!head -250 evaluation/evaluation_QA_system/dataset_filter/filter_data.py

#!/usr/bin/env python
# coding: utf-8

# # Filter dataset

# first we loop trough each training set for example BioASQ-trainingDataset2b.json and extract the pubmed IDS used to answers questions 

# In[1]:


import os
import json
import pandas as pd
from tqdm import tqdm

# Define the directories
json_dir = '~/Questions_answers_data/DATEN_RAG_PM4/trainings_sets'
csv_dir = os.path.expanduser(json_dir + '/csv')  # Ensure the path is expanded to the user's home directory

# Create the CSV directory if it doesn't exist
os.makedirs(csv_dir, exist_ok=True)

# Initialize a set to hold all unique PubMed IDs across files
all_pubmed_ids = set()

# List all JSON files in the directory
json_files = [f for f in os.listdir(os.path.expanduser(json_dir)) if f.endswith('.json')]  # Ensure the path is expanded

# Loop through files with a tqdm progress bar
for json_file in tqdm(json_files, desc="Processing JSON Files"):
    json_path = os.path.join(os.path.expanduser(json_dir), json_file)

    # Load JS

# BM25 Retrieval

In [38]:
!pip install -q rank_bm25

In [39]:
from rank_bm25 import BM25Okapi
from datasets import load_dataset
from tqdm import tqdm
import re

In [40]:
pubmed = load_dataset(
    "slinusc/PubMedAbstractsSubset",
    split="train",
    streaming=True
)

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

In [41]:
documents = []

for i, doc in enumerate(pubmed):

    documents.append(doc)

    if i == 9999:
        break

print("Number of documents:", len(documents))

Number of documents: 10000


In [42]:
corpus = []

for doc in documents:

    text = doc["title"] + " " + doc["abstract"]

    corpus.append(text)

In [43]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()

In [44]:
tokenized_corpus = [
    preprocess(doc)
    for doc in corpus
]

In [45]:
tokenized_corpus[0]

['demonstration',
 'of',
 'tumor',
 'inhibiting',
 'properties',
 'of',
 'a',
 'strongly',
 'immunostimulating',
 'lowmolecular',
 'weight',
 'substance',
 'comparative',
 'studies',
 'with',
 'ifosfamide',
 'on',
 'the',
 'immunolabile',
 'ds',
 'carcinosarcoma',
 'stimulation',
 'of',
 'the',
 'autoimmune',
 'activity',
 'for',
 'approx',
 '20',
 'days',
 'by',
 'ba',
 '1',
 'a',
 'n2cyanoethyleneurea',
 'novel',
 'prophylactic',
 'possibilities',
 'a',
 'report',
 'is',
 'given',
 'on',
 'the',
 'recent',
 'discovery',
 'of',
 'outstanding',
 'immunological',
 'properties',
 'in',
 'ba',
 '1',
 'n2cyanoethyleneurea',
 'having',
 'a',
 'low',
 'molecular',
 'mass',
 'm',
 '111104',
 'experiments',
 'in',
 '214',
 'ds',
 'carcinosarcoma',
 'bearing',
 'wistar',
 'rats',
 'have',
 'shown',
 'that',
 'ba',
 '1',
 'at',
 'a',
 'dosage',
 'of',
 'only',
 'about',
 '12',
 'percent',
 'ld50',
 '150',
 'mg',
 'kg',
 'and',
 'negligible',
 'lethality',
 '17',
 'percent',
 'results',
 'in',
 '

In [46]:
bm25 = BM25Okapi(tokenized_corpus)

In [47]:
documents[0]

{'title': '[Demonstration of tumor inhibiting properties of a strongly immunostimulating low-molecular weight substance. Comparative studies with ifosfamide on the immuno-labile DS carcinosarcoma. Stimulation of the autoimmune activity for approx. 20 days by BA 1, a N-(2-cyanoethylene)-urea. Novel prophylactic possibilities].',
 'abstract': 'A report is given on the recent discovery of outstanding immunological properties in BA 1 [N-(2-cyanoethylene)-urea] having a (low) molecular mass M = 111.104. Experiments in 214 DS carcinosarcoma bearing Wistar rats have shown that BA 1, at a dosage of only about 12 percent LD50 (150 mg kg) and negligible lethality (1.7 percent), results in a recovery rate of 40 percent without hyperglycemia and, in one test, of 80 percent with hyperglycemia. Under otherwise unchanged conditions the reference substance ifosfamide (IF) -- a further development of cyclophosphamide -- applied without hyperglycemia in its most efficient dosage of 47 percent LD50 (150 

In [48]:
tokenized_corpus[0][:30]

['demonstration',
 'of',
 'tumor',
 'inhibiting',
 'properties',
 'of',
 'a',
 'strongly',
 'immunostimulating',
 'lowmolecular',
 'weight',
 'substance',
 'comparative',
 'studies',
 'with',
 'ifosfamide',
 'on',
 'the',
 'immunolabile',
 'ds',
 'carcinosarcoma',
 'stimulation',
 'of',
 'the',
 'autoimmune',
 'activity',
 'for',
 'approx',
 '20',
 'days']

In [49]:
query = "What is the treatment for diabetes?"

In [50]:
tokenized_query = preprocess(query)

print(tokenized_query)

['what', 'is', 'the', 'treatment', 'for', 'diabetes']


In [51]:
scores = bm25.get_scores(tokenized_query)

In [52]:
import numpy as np

top_n = 5

top_indices = np.argsort(scores)[::-1][:top_n]

In [53]:
np.argsort(scores)

array([4270, 8711, 9030, ..., 5207, 6764, 6469])

In [54]:
for rank, idx in enumerate(top_indices, start=1):

    print("=" * 80)
    print(f"Rank {rank}")
    print(f"Score : {scores[idx]:.3f}")
    print(f"PMID  : {documents[idx]['PMID']}")
    print(f"Title : {documents[idx]['title']}")

Rank 1
Score : 20.537
PMID  : 137164
Title : [Incidence and manifestations of the heterozygosity of the gene causing diabetes].
Rank 2
Score : 19.322
PMID  : 143916
Title : Islet transplantation in genetically determined diabetes.
Rank 3
Score : 18.598
PMID  : 111165
Title : [Current status in transplantation of the endocrine pancreas in the treatment of diabetes mellitus 2. Transplantation of fetal pancreas].
Rank 4
Score : 18.264
PMID  : 83349
Title : The tumor-producing effect of automobile exhaust condensate and fractions thereof. Part III: mathematical-statistical evaluation of the test results.
Rank 5
Score : 18.114
PMID  : 99735
Title : [Enlarge indications for controlled respiration (author's transl)].


In [55]:
import re

STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were",
    "of", "to", "for", "in", "on", "at", "with",
    "what", "which", "who", "when", "where", "why",
    "how", "and", "or"
}

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)

    tokens = text.split()

    tokens = [
        word
        for word in tokens
        if word not in STOPWORDS
    ]

    return tokens

In [56]:
preprocess("What is the treatment for diabetes?")

['treatment', 'diabetes']

In [57]:
tokenized_corpus = [
    preprocess(doc)
    for doc in corpus
]

bm25 = BM25Okapi(tokenized_corpus)

In [58]:
query = "What is the treatment for diabetes?"

tokenized_query = preprocess(query)

scores = bm25.get_scores(tokenized_query)

top_indices = np.argsort(scores)[::-1][:5]

for rank, idx in enumerate(top_indices, start=1):

    print("="*70)
    print(f"Rank {rank}")
    print(f"Score : {scores[idx]:.2f}")
    print(f"PMID  : {documents[idx]['PMID']}")
    print(f"Title : {documents[idx]['title']}")

Rank 1
Score : 11.88
PMID  : 111165
Title : [Current status in transplantation of the endocrine pancreas in the treatment of diabetes mellitus 2. Transplantation of fetal pancreas].
Rank 2
Score : 11.08
PMID  : 143916
Title : Islet transplantation in genetically determined diabetes.
Rank 3
Score : 11.06
PMID  : 69935
Title : Familial studies of type-I and type-II idiopathic diabetes mellitus.
Rank 4
Score : 10.36
PMID  : 137164
Title : [Incidence and manifestations of the heterozygosity of the gene causing diabetes].
Rank 5
Score : 10.27
PMID  : 37498
Title : [Effect of alpha and beta receptor blockaders on the degree of glycemia, growth hormone content of blood and catecholamine excretion in insulin-dependent diabetes mellitus].


BIOBERT RETRIVAL

Question -> BioBERT Encoder -> Question Embedding -> Compare with Document Embeddings -> Top-k Documents

In [59]:
!pip install -q sentence-transformers

In [60]:
from sentence_transformers import SentenceTransformer
import numpy as np

# BioBERT Semantic Retrieval

In [61]:
from sentence_transformers import SentenceTransformer, models
import torch

In [62]:
device = "cuda" if torch.cuda.is_available() else "cpu"

word_embedding_model = models.Transformer(
    "dmis-lab/biobert-v1.1",
    max_seq_length=512
)

pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)

biobert = SentenceTransformer(
    modules=[word_embedding_model, pooling_model],
    device=device
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_694/1262816461.py:9: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  word_embedding_model.get_word_embedding_dimension(),
/usr/local/lib/python3.12/dist-packages/sentence_transformers/util/decorators.py:41: FutureWarning: The `pooling_mode_mean_tokens` argument(s) are deprecated. Please use `pooling_mode` instead.
  return func(*args, **kwargs)


In [63]:
sentence = "Metformin is used to treat diabetes."

embedding = biobert.encode(sentence)

print(embedding.shape)

(768,)


In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [65]:
corpus = []

for doc in documents:
    text = doc["title"] + " " + doc["abstract"]
    corpus.append(text)

print(corpus[0][:200])

[Demonstration of tumor inhibiting properties of a strongly immunostimulating low-molecular weight substance. Comparative studies with ifosfamide on the immuno-labile DS carcinosarcoma. Stimulation of


In [66]:
from sentence_transformers import SentenceTransformer, models
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

word_embedding_model = models.Transformer(
    "dmis-lab/biobert-v1.1",
    max_seq_length=512
)

pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
    pooling_mode_cls_token=False,
    pooling_mode_max_tokens=False
)

biobert = SentenceTransformer(
    modules=[word_embedding_model, pooling_model],
    device=device
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_694/1872715834.py:12: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  word_embedding_model.get_word_embedding_dimension(),
/usr/local/lib/python3.12/dist-packages/sentence_transformers/util/decorators.py:41: FutureWarning: The `pooling_mode_mean_tokens`, `pooling_mode_cls_token`, `pooling_mode_max_tokens` argument(s) are deprecated. Please use `pooling_mode` instead.


In [67]:
sentence = "Metformin is used to treat diabetes."

embedding = biobert.encode(sentence)

print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(768,)


In [68]:
test_embeddings = biobert.encode(
    corpus[:10],
    show_progress_bar=True,
    convert_to_numpy=True
)

print(test_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(10, 768)


In [70]:
print(embedding.shape)

(768,)


In [71]:
print(test_embeddings.shape)

(10, 768)


In [72]:
import numpy as np

document_embeddings = biobert.encode(
    corpus,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding Shape:", document_embeddings.shape)

# Save embeddings
np.save("biobert_embeddings_10k.npy", document_embeddings)

print("Embeddings saved successfully!")

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Embedding Shape: (10000, 768)
Embeddings saved successfully!


In [73]:
print(document_embeddings.shape)

(10000, 768)


In [74]:
document_embeddings[:2]

array([[ 0.02567428, -0.19570796, -0.15966555, ...,  0.2537041 ,
        -0.09315996,  0.00287652],
       [ 0.07904427,  0.00125216, -0.12614757, ...,  0.29271612,
         0.00890271,  0.03762016]], dtype=float32)

In [75]:
query = "What is the treatment for diabetes?"

query_embedding = biobert.encode(
    query,
    convert_to_numpy=True
)

print(query_embedding.shape)

(768,)


In [76]:
from sklearn.metrics.pairwise import cosine_similarity

In [77]:
similarities = cosine_similarity(
    [query_embedding],
    document_embeddings
)[0]

In [78]:
import numpy as np

top_k = 5

top_indices = np.argsort(similarities)[::-1][:top_k]

In [79]:
for rank, idx in enumerate(top_indices, start=1):

    print("=" * 80)
    print(f"Rank: {rank}")
    print(f"Similarity: {similarities[idx]:.4f}")
    print(f"PMID: {documents[idx]['PMID']}")
    print(f"Title: {documents[idx]['title']}")

Rank: 1
Similarity: 0.8336
PMID: 152625
Title: Some problems of the severely handicapped.
Rank: 2
Similarity: 0.8334
PMID: 59936
Title: Radiotherapy of metastases of mammary carcinoma.
Rank: 3
Similarity: 0.8329
PMID: 190109
Title: Options for care of the aged sick.
Rank: 4
Similarity: 0.8315
PMID: 106817
Title: Epilepsy.
Rank: 5
Similarity: 0.8315
PMID: 73362
Title: Fiber types and metabolic potentials of skeletal muscles in sedentary man and endurance runners.


In [80]:
count = 0

for doc in documents:
    text = (doc["title"] + " " + doc["abstract"]).lower()

    if "diabetes" in text:
        count += 1

print("Documents mentioning diabetes:", count)

Documents mentioning diabetes: 89
